In [4]:
import pandas as pd
from backtesting import Backtest, Strategy
from vnstock3 import Vnstock
import talib as ta

RSI_PERIOD = 14
RSI_OVERSOLD = 30
OBV_PERIOD = 5

In [11]:
 def calculate_first_mondays(dates):
        if not isinstance(dates, pd.DatetimeIndex):
            dates = pd.DatetimeIndex(dates)
        dates_series = pd.Series(dates, index=dates)
        mondays = dates_series[dates_series.dt.dayofweek == 0]
        first_mondays = mondays.groupby([mondays.dt.year, mondays.dt.month]).first()
        return set(first_mondays)
     
class DCA(Strategy):
    average_monthly_income_vnd = 1000
    investment_percentage = 0.10  
    fund = 0

    def init(self):
        close = self.data.Close.astype(float)  
        volume = self.data.Volume.astype(float)  

        # Calculate RSI and OBV
        self.rsi = self.I(ta.RSI, close, timeperiod=RSI_PERIOD)
        self.obv = self.I(ta.OBV, close, volume)
        self.obv_slope = self.I(pd.Series(self.obv).diff, periods=OBV_PERIOD)
        self.previous_rsi = self.I(pd.Series(self.rsi).shift, 1)
        self.first_mondays = calculate_first_mondays(self.data.index)

    def next(self):
        today = self.data.index[-1]
        self.data.Close[-1] = self.data.Close[-1] / 10
        if today in self.first_mondays:
            self.fund += self.average_monthly_income_vnd * self.investment_percentage

        # Check for buy signal: rsi cắt và obv dương
        if (self.previous_rsi[-1] < RSI_OVERSOLD and
            self.rsi[-1] >= RSI_OVERSOLD and
            self.obv_slope[-1] > 0):
            share_price = self.data.Close[-1]
            shares_to_buy = self.fund // share_price
            shares_to_buy = (shares_to_buy // 100) * 100
            if shares_to_buy > 0:
                self.buy(size=shares_to_buy)
                self.fund -= share_price * shares_to_buy
                
def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol,source='VCI').quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)
    stock_data.index = stock_data.index.normalize()
    stock_data = stock_data.dropna()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()
    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]

    print(f"Results for {stock_symbol}:")
    print(trades)
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# List of stock symbols
stock_symbols = ['FPT','MWG','E1VFVN30'] 

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)

2024-10-26 17:54:55,548 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-10-26 17:54:56,591 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for FPT:
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice     PnL  ReturnPct  \
0   400       767     1251       5.181      8.316  1254.0   0.605096   
1   700       304     1251       1.961      8.316  4448.5   3.240694   

   EntryTime   ExitTime  Duration  
0 2022-01-25 2024-01-03  708 days  
1 2020-03-25 2024-01-03 1379 days  
Total investment: 3445.1
Current Shares: 1100
Current Equity: 9233.4
RoR: 168.0154422222867
--------------------------------------------------


2024-10-26 17:54:57,218 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for MWG:
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice     PnL  ReturnPct  \
0   800       941     1251       5.611      4.286 -1060.0  -0.236143   

   EntryTime   ExitTime Duration  
0 2022-10-10 2024-01-03 450 days  
Total investment: 4488.8
Current Shares: 800
Current Equity: 3428.7999999999997
RoR: -23.614328996613803
--------------------------------------------------
Results for E1VFVN30:
   Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0   300       941     1247         1.8      1.969   50.7   0.093889   
1  1700       831     1247         2.3      1.969 -562.7  -0.143913   

   EntryTime   ExitTime Duration  
0 2022-10-14 2024-01-03 446 days  
1 2022-05-10 2024-01-03 603 days  
Total investment: 4450.0
Current Shares: 2000
Current Equity: 3989.9999999999995
RoR: -10.337078651685404
--------------------------------------------------


In [12]:
 def calculate_first_mondays(dates):
        if not isinstance(dates, pd.DatetimeIndex):
            dates = pd.DatetimeIndex(dates)
        dates_series = pd.Series(dates, index=dates)
        mondays = dates_series[dates_series.dt.dayofweek == 0]
        first_mondays = mondays.groupby([mondays.dt.year, mondays.dt.month]).first()
        return set(first_mondays)
     
class DCA(Strategy):
    average_monthly_income_vnd = 1000
    investment_percentage = 0.10  
    fund = 0

    def init(self):
        close = self.data.Close.astype(float)  
        volume = self.data.Volume.astype(float)  

        # Calculate RSI and OBV
        self.rsi = self.I(ta.RSI, close, timeperiod=RSI_PERIOD)
        self.obv = self.I(ta.OBV, close, volume)
        self.obv_slope = self.I(pd.Series(self.obv).diff, periods=OBV_PERIOD)
        self.previous_rsi = self.I(pd.Series(self.rsi).shift, 1)
        self.first_mondays = calculate_first_mondays(self.data.index)

    def next(self):
        today = self.data.index[-1]
        self.data.Close[-1] = self.data.Close[-1] / 10
        if today in self.first_mondays:
            self.fund += self.average_monthly_income_vnd * self.investment_percentage

        # Check for buy signal: rsi trên 30
        if (self.rsi[-1] > RSI_OVERSOLD and 
            self.obv_slope[-1] > 0):
            share_price = self.data.Close[-1]
            shares_to_buy = self.fund // share_price
            shares_to_buy = (shares_to_buy // 100) * 100
            if shares_to_buy > 0:
                self.buy(size=shares_to_buy)
                self.fund -= share_price * shares_to_buy
                
def run_backtest(stock_symbol):
    # Fetch stock data
    stock_data = Vnstock().stock(symbol=stock_symbol,source='VCI').quote.history(start='2019-01-01', end='2024-01-04')
    stock_data = stock_data.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    stock_data.set_index('time', inplace=True)
    stock_data.index = pd.to_datetime(stock_data.index)
    stock_data.index = stock_data.index.normalize()
    stock_data = stock_data.dropna()

    # Run the backtest
    bt = Backtest(
        stock_data,
        DCA,
        trade_on_close=True,
    )
    stats = bt.run()
    #bt.plot(filename=f'{stock_symbol}')
    
    # Calculate investment details
    trades = stats["_trades"]
    price_paid = trades["Size"] * trades["EntryPrice"]
    total_invested = price_paid.sum()
    current_shares = trades["Size"].sum()
    current_equity = current_shares * stock_data.Close.iloc[-1]

    print(f"Results for {stock_symbol}:")
    print(trades)
    print("Total investment:", total_invested)
    print("Current Shares:", current_shares)
    print("Current Equity:", current_equity)
    print("RoR:", ((current_equity - total_invested) / total_invested)*100)
    print("-" * 50)

# List of stock symbols
stock_symbols = ['FPT','MWG','E1VFVN30'] 

# Run backtest for each stock
for symbol in stock_symbols:
    run_backtest(symbol)

2024-10-26 17:55:19,691 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
2024-10-26 17:55:20,398 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for FPT:
    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0    100      1212     1251       7.997      8.316   31.9   0.039890   
1    100      1043     1251       5.914      8.316  240.2   0.406155   
2    100       923     1251       6.163      8.316  215.3   0.349343   
3    100       791     1251       5.618      8.316  269.8   0.480242   
4    100       667     1251       5.684      8.316  263.2   0.463054   
5    100       562     1251       4.060      8.316  425.6   1.048276   
6    100       483     1251       2.841      8.316  547.5   1.927138   
7    100       419     1251       2.525      8.316  579.1   2.293465   
8    100       351     1251       2.430      8.316  588.6   2.422222   
9    100       311     1251       2.025      8.316  629.1   3.106667   
10   100       271     1251       2.272      8.316  604.4   2.660211   
11   100       234     1251       2.366      8.316  595.0   2.514793   
12   100       168     1251       2.294      8.

2024-10-26 17:55:21,087 - vnstock3.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS


Results for MWG:
    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    PnL  ReturnPct  \
0    100      1185     1251       5.149      4.286  -86.3  -0.167605   
1    100      1082     1251       3.719      4.286   56.7   0.152460   
2    100      1011     1251       4.230      4.286    5.6   0.013239   
3    100       941     1251       5.611      4.286 -132.5  -0.236143   
4    100       801     1251       6.585      4.286 -229.9  -0.349127   
5    100       667     1251       5.418      4.286 -113.2  -0.208933   
6    100       547     1251       4.305      4.286   -1.9  -0.004413   
7    100       460     1251       3.465      4.286   82.1   0.236941   
8    100       394     1251       2.432      4.286  185.4   0.762336   
9    100       332     1251       2.643      4.286  164.3   0.621642   
10   100       287     1251       3.407      4.286   87.9   0.257998   
11   100       210     1251       3.986      4.286   30.0   0.075263   
12   100       119     1251       3.030      4.